<a href="https://colab.research.google.com/github/profliuhao/CSIT599/blob/main/CSIT599_Online_lab3_frameworks_pytorch_tensorflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3 — One Model, Two Frameworks (Keras/TensorFlow vs. PyTorch)

**Module 3: Deep Learning Frameworks**

In this lab you will implement the **same small CNN twice** — once with the high-level
**Keras** API in TensorFlow and once in **PyTorch** — train both on **Fashion-MNIST**,
then evaluate properly with **k-fold cross-validation** and standard **metrics**.

The point is not that one framework "wins": it's to see that the *concepts* (layers,
losses, optimizers, training loops) are identical, while the *ergonomics* differ.

## What you will do
You only need to fill in **six** short pieces of code, each marked with a `TODO`:
  1. define the CNN with **Keras** (`Sequential`)
  2. **compile and fit** the Keras model
  3. define the same CNN in **PyTorch** (`nn.Module`)
  4. write one **training epoch** for PyTorch
  5. run **3-fold cross-validation** of the PyTorch model
  6. compute **precision / recall / F1 and a confusion matrix**

Everything else — data loading, plotting, and the framework comparison — is provided.

## How to use this file
* Recommended: **Google Colab** (both TensorFlow and PyTorch are pre-installed there).
* GPU helps but is not required — we train on a subset so CPU finishes in a few minutes.
* Fill in each `TODO`, then run cells top to bottom.

## Setup — imports and configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import KFold
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
EPOCHS = 4
BATCH_SIZE = 128
TRAIN_SUBSET = 12_000     # of 60k, so everything runs fast
TEST_SUBSET = 2_000

np.random.seed(SEED); tf.random.set_seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("TF:", tf.__version__, "| Torch:", torch.__version__, "| device:", device)

## Section 1 — The data *(provided)*

Fashion-MNIST: 70,000 grayscale 28x28 images of clothing items, 10 classes — a drop-in
replacement for MNIST that is a bit harder. We scale pixels to [0, 1] and keep a subset.

In [ ]:
(Xtr, ytr), (Xte, yte) = keras.datasets.fashion_mnist.load_data()
CLASSES = ["t-shirt", "trouser", "pullover", "dress", "coat",
           "sandal", "shirt", "sneaker", "bag", "ankle boot"]

rng = np.random.RandomState(SEED)
itr = rng.choice(len(Xtr), TRAIN_SUBSET, replace=False)
ite = rng.choice(len(Xte), TEST_SUBSET, replace=False)
Xtr, ytr, Xte, yte = Xtr[itr], ytr[itr], Xte[ite], yte[ite]

Xtr = (Xtr / 255.0).astype("float32")
Xte = (Xte / 255.0).astype("float32")
print("train:", Xtr.shape, " test:", Xte.shape)

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, img, lab in zip(axes, Xtr[:8], ytr[:8]):
    ax.imshow(img, cmap="gray"); ax.set_title(CLASSES[lab], fontsize=8); ax.axis("off")
plt.show()

## Section 2 — The Keras way

**TODO 1:** build this architecture with `keras.Sequential`:

```
Input (28, 28, 1)
Conv2D(32, 3, padding="same", activation="relu") -> MaxPooling2D(2)
Conv2D(64, 3, padding="same", activation="relu") -> MaxPooling2D(2)
Flatten -> Dropout(0.25) -> Dense(128, activation="relu") -> Dense(10)
```

(No softmax on the last layer — we use a from-logits loss, mirroring PyTorch.)

**TODO 2:** `compile` with the Adam optimizer and
`SparseCategoricalCrossentropy(from_logits=True)` plus the `"accuracy"` metric, then
`fit` on `Xtr`/`ytr` for `EPOCHS` epochs with `batch_size=BATCH_SIZE` and
`validation_split=0.1`, storing the result in `history`.

In [ ]:
def build_keras_model():
    # ===== TODO 1: return the keras.Sequential model described above =====
    raise NotImplementedError("TODO 1: build the Keras model, then delete this line.")

keras_model = build_keras_model()
keras_model.summary()

In [ ]:
# ===== TODO 2: compile and fit (validation_split=0.1), assign to `history` =====
history = None
raise NotImplementedError("TODO 2: compile and fit the Keras model, then delete this line.")

keras_test_acc = keras_model.evaluate(Xte[..., None], yte, verbose=0)[1]
print(f"Keras test accuracy: {keras_test_acc:.3f}")

## Section 3 — The PyTorch way

**TODO 3:** implement the *same* architecture as an `nn.Module`. PyTorch uses
channels-first tensors `(batch, 1, 28, 28)`.

**TODO 4:** write `train_one_epoch`: loop over the loader and for each batch do
`zero_grad → forward → loss → backward → step`; return the mean batch loss.
Note how much Keras's `fit()` was doing for you!

In [ ]:
class TorchCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # ===== TODO 3a: define the layers (mirror the Keras model) =====
        pass  # replace with conv1, conv2, pool, dropout, fc1, fc2

    def forward(self, x):
        # ===== TODO 3b: conv1->relu->pool, conv2->relu->pool, flatten, dropout, fc1->relu, fc2 =====
        raise NotImplementedError("TODO 3: implement the forward pass, then delete this line.")

def make_loaders(X, y, Xval, yval, batch_size=BATCH_SIZE):
    tr = TensorDataset(torch.tensor(X)[:, None], torch.tensor(y, dtype=torch.long))
    va = TensorDataset(torch.tensor(Xval)[:, None], torch.tensor(yval, dtype=torch.long))
    return (DataLoader(tr, batch_size=batch_size, shuffle=True),
            DataLoader(va, batch_size=256, shuffle=False))

criterion = nn.CrossEntropyLoss()

def train_one_epoch(net, loader, optimizer):
    """One pass over `loader`; returns mean batch loss (float)."""
    net.train()
    losses = []
    # ===== TODO 4: the standard PyTorch training loop body =====
    raise NotImplementedError("TODO 4: implement the training epoch, then delete this line.")

@torch.no_grad()
def accuracy(net, loader):
    net.eval()
    correct = total = 0
    for xb, yb in loader:
        preds = net(xb.to(device)).argmax(1).cpu()
        correct += (preds == yb).sum().item(); total += len(yb)
    return correct / total

In [ ]:
# Train the PyTorch model with the same data split Keras used (provided).
n_val = int(0.1 * len(Xtr))
tr_loader, va_loader = make_loaders(Xtr[n_val:], ytr[n_val:], Xtr[:n_val], ytr[:n_val])
te_loader = make_loaders(Xtr[:1], ytr[:1], Xte, yte)[1]

torch_model = TorchCNN().to(device)
optimizer = torch.optim.Adam(torch_model.parameters(), lr=1e-3)
torch_hist = {"loss": [], "val_acc": []}
for epoch in range(1, EPOCHS + 1):
    loss = train_one_epoch(torch_model, tr_loader, optimizer)
    vacc = accuracy(torch_model, va_loader)
    torch_hist["loss"].append(loss); torch_hist["val_acc"].append(vacc)
    print(f"epoch {epoch}/{EPOCHS}  loss={loss:.4f}  val_acc={vacc:.3f}")

torch_test_acc = accuracy(torch_model, te_loader)
print(f"PyTorch test accuracy: {torch_test_acc:.3f}")

In [ ]:
# Side-by-side learning curves (provided).
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(history.history["loss"], label="Keras")
ax[0].plot(torch_hist["loss"], label="PyTorch")
ax[0].set_title("training loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(history.history["val_accuracy"], label="Keras")
ax[1].plot(torch_hist["val_acc"], label="PyTorch")
ax[1].set_title("validation accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.show()

## Section 4 — Honest evaluation: k-fold cross-validation

A single train/test split can be lucky or unlucky. **Cross-validation** trains the model
k times, each time holding out a different fold for validation, and reports the spread.

**TODO 5:** complete the loop using `KFold(n_splits=3, shuffle=True, random_state=SEED)`
over `Xcv`/`ycv`: for each `(train_index, val_index)` pair, build fresh loaders and a
fresh `TorchCNN`, train for 2 epochs with `train_one_epoch`, then record
`accuracy(net, val_loader)` in `fold_accs`.

In [ ]:
Xcv, ycv = Xtr[:6000], ytr[:6000]   # smaller slice so 3 folds stay fast
fold_accs = []

kf = KFold(n_splits=3, shuffle=True, random_state=SEED)
for fold, (train_index, val_index) in enumerate(kf.split(Xcv), start=1):
    # ===== TODO 5: build loaders/model, train 2 epochs, append val accuracy =====
    acc = None
    raise NotImplementedError("TODO 5: implement one cross-validation fold, then delete this line.")
    fold_accs.append(acc)
    print(f"fold {fold}: val_acc={acc:.3f}")

print(f"\ncross-validation accuracy: {np.mean(fold_accs):.3f} ± {np.std(fold_accs):.3f}")

## Section 5 — Beyond accuracy: per-class metrics

Accuracy hides *which* classes a model confuses. **TODO 6:**
1. get predictions of `torch_model` on the test set (argmax over logits, no gradients),
2. print `classification_report(yte, preds, target_names=CLASSES)`,
3. compute `cm = confusion_matrix(yte, preds)` (the plot below is provided).

In [ ]:
# ===== TODO 6: predictions, classification report, confusion matrix =====
preds, cm = None, None
raise NotImplementedError("TODO 6: compute predictions, report, and confusion matrix, then delete this line.")

plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap="Blues")
plt.xticks(range(10), CLASSES, rotation=45, ha="right"); plt.yticks(range(10), CLASSES)
plt.xlabel("predicted"); plt.ylabel("true"); plt.title("Confusion matrix (PyTorch model)")
plt.colorbar(); plt.tight_layout(); plt.show()

## Section 6 — What did each framework do for you? *(provided — read it!)*

| concern              | Keras                                  | PyTorch                          |
|----------------------|----------------------------------------|----------------------------------|
| model definition     | `Sequential([...])` declarative        | `nn.Module` subclass, explicit   |
| training loop        | `model.fit()` (hidden)                 | you write it (full control)      |
| validation           | `validation_split=0.1` argument        | you build the split & loop       |
| logging/metrics      | built into `fit`/`evaluate`            | you collect them yourself        |
| flexibility          | callbacks/custom layers when needed    | everything is just Python        |

Notice the *test accuracies are nearly identical* — the architecture and data determine
performance, not the framework. Keep this in mind for the M3 discussion on choosing a
framework, and for HW 2 you may use either one.

## Checklist before you submit

* [ ] All cells run top-to-bottom without errors, outputs visible.
* [ ] Keras and PyTorch models both reach **~88%+** test accuracy.
* [ ] Cross-validation prints a mean ± std over 3 folds.
* [ ] Classification report and confusion matrix are shown.

**Submit your completed notebook (.ipynb) with all outputs visible.**